# StellarStats — 01: Exploratory Data Analysis

## Dataset
- Source: HYG Star Database v4.2
- Raw stars: 119,626
- Stars after cleaning: 107,859 (9.84% removed)

## Key findings
- Luminosity is extremely right-skewed — log scale required
- Temperature derived from colour index (B-V) using standard formula
- Distance sentinel values (100,000 pc) identified and removed

In [1]:
# Core libraries for data manipulation and visualization
import pandas as pd        # DataFrame operations and CSV loading
import numpy as np         # Numerical computation and array operations
import matplotlib.pyplot as plt  # Base plotting library
import seaborn as sns      # Statistical visualizations built on matplotlib

In [2]:
# Load the raw HYG Star Database v4.2 into a DataFrame
df = pd.read_csv('../data/hyg_v42.csv')

In [3]:
# Inspect dataset dimensions (rows, columns)
df.shape

(119626, 37)

In [4]:
# List all 37 column names to understand available features
df.columns.tolist()

['id',
 'hip',
 'hd',
 'hr',
 'gl',
 'bf',
 'proper',
 'ra',
 'dec',
 'dist',
 'pmra',
 'pmdec',
 'rv',
 'mag',
 'absmag',
 'spect',
 'ci',
 'x',
 'y',
 'z',
 'vx',
 'vy',
 'vz',
 'rarad',
 'decrad',
 'pmrarad',
 'pmdecrad',
 'bayer',
 'flam',
 'con',
 'comp',
 'comp_primary',
 'base',
 'lum',
 'var',
 'var_min',
 'var_max']

In [5]:
# Preview first 5 rows — row 0 is our Sun (Sol) with lum = 1.0 as expected
df.head()

,id,hip,hd,hr,gl,bf,proper,ra,dec,dist,...,bayer,flam,con,comp,comp_primary,base,lum,var,var_min,var_max
0,0,NaN,NaN,NaN,NaN,NaN,Sol,0.000000,0.000000,0.0000,...,NaN,NaN,NaN,1,0,NaN,1.000000,NaN,NaN,NaN
1,1,1.0,224700.0,NaN,NaN,NaN,NaN,0.000060,1.089009,219.7802,...,NaN,NaN,Psc,1,1,NaN,9.638290,NaN,NaN,NaN
2,2,2.0,224690.0,NaN,NaN,NaN,NaN,0.000283,-19.498840,47.9616,...,NaN,NaN,Cet,1,2,NaN,0.392283,NaN,NaN,NaN
3,3,3.0,224699.0,NaN,NaN,NaN,NaN,0.000335,38.859279,442.4779,...,NaN,NaN,And,1,3,NaN,386.901132,NaN,NaN,NaN
4,4,4.0,224707.0,NaN,NaN,NaN,NaN,0.000569,-51.893546,134.2282,...,NaN,NaN,Phe,1,4,NaN,9.366989,NaN,NaN,NaN


In [6]:
# Check data types per column — critical before any analysis
# Numeric columns (float64/int64) are ready to use
# String columns (str) need special handling
df.dtypes

id                int64
hip             float64
hd              float64
hr              float64
gl                  str
bf                  str
proper              str
ra              float64
dec             float64
dist            float64
pmra            float64
pmdec           float64
rv              float64
mag             float64
absmag          float64
spect               str
ci              float64
x               float64
y               float64
z               float64
vx              float64
vy              float64
vz              float64
rarad           float64
decrad          float64
pmrarad         float64
pmdecrad        float64
bayer               str
flam            float64
con                 str
comp              int64
comp_primary      int64
base                str
lum             float64
var                 str
var_min         float64
var_max         float64
dtype: object

In [7]:
# Count missing values per column
# isnull() returns True/False per cell, sum() counts the Trues
df.isnull().sum()

id                   0
hip               1675
hd               20741
hr              110585
gl              115825
bf              116527
proper          119127
ra                   0
dec                  0
dist                 0
pmra                 0
pmdec                0
rv                   0
mag                  0
absmag               0
spect             3048
ci                1891
x                    0
y                    0
z                    0
vx                   0
vy                   0
vz                   0
rarad                0
decrad               0
pmrarad              0
pmdecrad             0
bayer           118089
flam            116889
con                  1
comp                 0
comp_primary         0
base            118540
lum                  0
var             113634
var_min         102635
var_max         102635
dtype: int64

In [8]:
# Summary statistics for all numeric columns
# Key finding: dist max = 100,000 pc — a sentinel placeholder value, not real data
df.describe()

,id,hip,hd,hr,ra,dec,dist,pmra,pmdec,rv,...,rarad,decrad,pmrarad,pmdecrad,flam,comp,comp_primary,lum,var_min,var_max
count,119626.000000,117951.000000,98885.000000,9041.000000,119626.000000,119626.000000,119626.000000,119626.000000,119626.000000,119626.000000,...,119626.000000,119626.000000,1.196260e+05,1.196260e+05,2737.000000,119626.000000,119626.000000,1.196260e+05,16991.000000,16991.000000
mean,59813.164805,59169.527109,114357.226253,4563.897578,12.094713,-1.986047,8772.285030,-1.306801,-19.329857,-0.276455,...,3.166388,-0.034663,-6.400491e-09,-9.363022e-08,37.318597,1.004815,59641.955679,3.565260e+05,9.501549,9.258892
std,34533.826601,34171.706401,74176.412852,2632.048273,6.887466,40.964667,27890.666828,118.175058,112.502380,13.904226,...,1.803134,0.714968,5.729060e-07,5.466507e-07,27.744027,0.073896,34441.658163,3.341375e+06,1.781276,1.742416
min,0.000000,1.000000,1.000000,1.000000,0.000000,-89.782428,0.000000,-4432.650000,-5813.000000,-386.900000,...,0.000000,-1.566999,-2.149009e-05,-2.818222e-05,1.000000,1.000000,0.000000,1.225745e-06,-1.333000,-1.523000
25%,29906.250000,29564.500000,46723.000000,2283.000000,6.217265,-36.422363,115.074800,-15.460000,-22.397500,0.000000,...,1.627676,-0.635690,-7.495220e-08,-1.085861e-07,15.000000,1.000000,29815.250000,4.746790e+00,8.525500,8.243000
50%,59813.500000,59172.000000,110358.000000,4566.000000,12.127026,-1.640078,213.675200,-1.680000,-5.760000,0.000000,...,3.174848,-0.028625,-8.144870e-09,-2.792527e-08,32.000000,1.000000,59634.500000,2.197860e+01,9.849000,9.646000
75%,89719.750000,88762.500000,175823.000000,6848.000000,18.115429,31.514932,392.156900,12.180000,3.770000,0.000000,...,4.742608,0.550039,5.905031e-08,1.827748e-08,55.000000,1.000000,89462.750000,7.670082e+01,10.707000,10.492000
max,119630.000000,120404.000000,358431.000000,9110.000000,23.998594,89.569427,100000.000000,6767.260000,9999.990000,471.000000,...,6.282817,1.563281,3.280860e-05,5.006637e-05,140.000000,3.000000,119630.000000,4.092607e+08,14.902000,13.702000


In [9]:
# Identify stars with placeholder distance of 100,000 pc (unknown distance)
# These are not real measurements and must be excluded from distance-based analysis
(df['dist'] == 100000).sum()

np.int64(10225)

In [10]:
# Summary statistics for our core analysis columns only
# Reveals extreme right skew in lum — log scale will be required
df[['lum', 'ci', 'absmag', 'dist', 'spect']].describe()

,lum,ci,absmag,dist
count,1.196260e+05,117735.000000,119626.000000,119626.000000
mean,3.565260e+05,0.711516,0.990741,8772.285030
std,3.341375e+06,0.493218,4.353396,27890.666828
min,1.225745e-06,-0.400000,-16.680000,0.000000
25%,4.746790e+00,0.348500,0.138000,115.074800
50%,2.197860e+01,0.616000,1.495000,213.675200
75%,7.670082e+01,1.083000,3.159000,392.156900
max,4.092607e+08,5.460000,19.629000,100000.000000


In [11]:
# Create cleaned working DataFrame by filtering out:
# - dist <= 0 (physically impossible distances)
# - dist >= 100,000 pc (sentinel placeholder values)
# - missing colour index (needed for temperature derivation)
# - lum <= 0 (physically impossible luminosities)
# .copy() ensures df_clean is fully independent from df
df_clean = df[
    (df['dist'] > 0) &
    (df['dist'] < 100000) &
    (df['ci'].notna()) &
    (df['lum'] > 0)
].copy()

print(df_clean.shape)

(107859, 37)


In [12]:
# Quantify data lost during cleaning
# :.2f formats percentage to 2 decimal places
# :, adds thousands separator for readability
lost = len(df) - len(df_clean)
pct_lost = (lost / len(df)) * 100
print(f"Rows lost: {lost}")
print(f"Percentage lost: {pct_lost:.2f}%")

Rows lost: 11767
Percentage lost: 9.84%


In [13]:
# Derive surface temperature from B-V colour index using standard formula:
# T = 9600 / (ci + 0.92)
# Validated against known solar temperature (~5778K at ci ≈ 0.65)
df_clean['temp'] = 9600 / (df_clean['ci'] + 0.92)

In [14]:
# Verify temperature derivation looks physically sensible
# Lower ci = hotter star = higher temperature (inverse relationship)
df_clean[['ci', 'temp']].head(10)

,ci,temp
1,0.482,6847.360913
2,0.999,5002.605524
3,-0.019,10654.827969
4,0.370,7441.860465
5,0.902,5268.935236
6,1.336,4255.319149
7,0.740,5783.132530
8,1.102,4747.774481
9,1.067,4831.404127
10,0.489,6813.342796


In [15]:
# Final summary of cleaned dataset ranges
# :.0f = no decimal places, :.6f = 6 decimal places, :, = thousands separator
print(f"Total stars in dataset: {len(df):,}")
print(f"Stars after cleaning: {len(df_clean):,}")
print(f"Temperature range: {df_clean['temp'].min():.0f}K - {df_clean['temp'].max():.0f}K")
print(f"Luminosity range: {df_clean['lum'].min():.6f} - {df_clean['lum'].max():.0f} (solar luminosities)")
print(f"Colour index range: {df_clean['ci'].min():.3f} - {df_clean['ci'].max():.3f}")

Total stars in dataset: 119,626
Stars after cleaning: 107,859
Temperature range: 1505K - 18462K
Luminosity range: 0.000003 - 67484 (solar luminosities)
Colour index range: -0.400 - 5.460


In [16]:
# Save cleaned DataFrame to CSV for use in all subsequent notebooks
# index=False prevents pandas from saving the row numbers as an extra column
df_clean.to_csv('../data/hyg_clean.csv', index=False)
print("Saved successfully")

Saved successfully
